# Generate Tree Sequences for each population

## Load 1000 Genomes Tree Sequence (.tsz)

This follows the same loading pattern used in `bim-paper/1000GenomesProject/tools1kg.py` (`tskit.load(...)`).

Samples are selected from population metadata before subsetting to the LCT region. 
Target populations here are CER, GBR, TSI, and JPT (CER is treated as CEU if CEU exists in metadata).

In [1]:
from pathlib import Path
import json
import pandas as pd
import tskit
import numpy as np

trees_dir = Path('trees')
results_dir = Path('results')
trees_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)


In [2]:
def path_1000g(chr_no: int, base_dir: Path = Path('.')) -> Path:
    return base_dir / f"1kg_chr{chr_no}.trees.tsz"

CHR = 2
ts_path = path_1000g(CHR)

# Same primary approach as bim-paper: tskit.load(path)
try:
    ts = tskit.load(str(ts_path))
except Exception:
    # Fallback for environments where .tsz requires explicit decompression
    import tszip
    ts = tszip.decompress(str(ts_path))

ts

In [3]:
# Build population metadata table (like bim-paper 1000G example)
pop_meta = pd.DataFrame([json.loads(ts.population(i).metadata) for i in range(ts.num_populations)])
pop_meta["pop_id"] = np.arange(ts.num_populations)

target_pops = ["CEU", "GBR", "TSI", "JPT", "KHV", "CHB"]
num_samples = 52

def resolve_pop_name(name: str) -> str:
    if name in set(pop_meta["name"]):
        return name
    raise ValueError("Population not found in metadata: " + name + ", available: " + ", ".join(pop_meta["name"].astype(str)))

lct_interval = [ (90_000_000, 160_000_000) ] # Region with LCT and surroundings

for pop in target_pops:
    pop_lookup = resolve_pop_name(pop)
    pop_id = int(pop_meta.loc[pop_meta["name"] == pop_lookup, "pop_id"].iloc[0])
    pop_samples = ts.samples(pop_id)

    rng = np.random.default_rng(1)
    kept = rng.choice(pop_samples, size=num_samples, replace=False)

    ts_lct = ts.keep_intervals(lct_interval, simplify=False)
    ts_lct_50 = ts_lct.simplify(samples=kept)
    ts_lct_50.dump(trees_dir / f"chr2_{num_samples}samples_{pop}.trees")
    print(f"Saved chr2_{num_samples}samples_{pop}.trees (pop={pop_lookup})")


Saved chr2_52samples_CEU.trees (pop=CEU)
Saved chr2_52samples_GBR.trees (pop=GBR)
Saved chr2_52samples_TSI.trees (pop=TSI)
Saved chr2_52samples_JPT.trees (pop=JPT)
Saved chr2_52samples_KHV.trees (pop=KHV)
Saved chr2_52samples_CHB.trees (pop=CHB)


In [4]:
pop_meta

,description,name,super_population,pop_id
0,"Han Chinese in Beijing, China",CHB,EAS,0
1,"Japanese in Tokyo, Japan",JPT,EAS,1
2,Southern Han Chinese,CHS,EAS,2
3,"Chinese Dai in Xishuangbanna, China",CDX,EAS,3
4,"Kinh in Ho Chi Minh City, Vietnam",KHV,EAS,4
5,Utah Residents (CEPH) with Northern and Wester...,CEU,EUR,5
6,Toscani in Italia,TSI,EUR,6
7,Finnish in Finland,FIN,EUR,7
8,British in England and Scotland,GBR,EUR,8
9,Iberian Population in Spain,IBS,EUR,9


## Convert to Newick format for analysis in R

In [5]:
import csv
import os
import numpy as np
import tskit

def jitter_within_generations(ts: tskit.TreeSequence, eps: float = 1e-9) -> tskit.TreeSequence:
    tables = ts.dump_tables()
    times = tables.nodes.time.astype(float).copy()

    is_sample = np.zeros(len(times), dtype=bool)
    is_sample[ts.samples()] = True

    jitter = np.zeros(len(times), dtype=float)

    for t in np.unique(times):
        idx = np.where(times == t)[0]
        idx = [i for i in idx if not is_sample[i]]
        if len(idx) > 1:
            for rank, node_id in enumerate(sorted(idx)):
                jitter[node_id] = eps * rank
                times[node_id] = t + jitter[node_id]

    tables.nodes.set_columns(
        flags=tables.nodes.flags,
        time=times,
        population=tables.nodes.population,
        individual=tables.nodes.individual,
        metadata=tables.nodes.metadata,
        metadata_offset=tables.nodes.metadata_offset,
    )

    mut_times = tables.mutations.time.astype(float).copy()
    mut_nodes = tables.mutations.node
    known = mut_times != tskit.UNKNOWN_TIME
    mut_times[known] = mut_times[known] + jitter[mut_nodes[known]]

    tables.mutations.set_columns(
        site=tables.mutations.site,
        node=mut_nodes,
        derived_state=tables.mutations.derived_state,
        derived_state_offset=tables.mutations.derived_state_offset,
        parent=tables.mutations.parent,
        time=mut_times,
        metadata=tables.mutations.metadata,
        metadata_offset=tables.mutations.metadata_offset,
    )

    tables.sort()
    return tables.tree_sequence()


def to_single_newick(tree):
    # tskit can have multiple roots; wrap root-specific newicks under a synthetic root.
    if tree.has_single_root:
        return tree.as_newick()

    parts = []
    for r in tree.roots:
        s = tree.as_newick(root=r)
        if s.endswith(";"):
            s = s[:-1]
        parts.append(s)
    return "(" + ",".join(parts) + ");"


In [6]:
target_pops = ["CEU", "GBR", "TSI", "JPT", "KHV", "CHB"]

for pop in target_pops:
    input_path = trees_dir / f"chr2_{num_samples}samples_{pop}.trees"
    output_tree_path = trees_dir / f"chr2_{num_samples}samples_{pop}.tree"
    output_csv_path = results_dir / f"chr2_{num_samples}samples_{pop}_breaks.csv"
    stem = f"chr2_{num_samples}samples_{pop}"

    ts_for_export = tskit.load(input_path)
    ts_for_export = jitter_within_generations(ts_for_export, eps=1e-9)

    # IMPORTANT: tskit reuses a mutable Tree object while iterating.
    # Snapshot all values/newick immediately; do not store Tree object references.
    records = []
    for original_index, tree in enumerate(ts_for_export.trees()):
        if tree.num_edges == 0:
            continue
        left, right = tree.interval.left, tree.interval.right
        records.append({
            "tree_index": original_index,
            "left": float(left),
            "right": float(right),
            "mid": float(0.5 * (left + right)),
            "newick": to_single_newick(tree),
        })

    if len(records) == 0:
        raise RuntimeError(f"No informative trees found for {pop} (all trees had num_edges == 0).")

    with open(output_tree_path, "w") as f:
        for rec in records:
            f.write(rec["newick"] + "\n")

    with open(output_csv_path, "w", newline="") as cf:
        writer = csv.writer(cf)
        writer.writerow(["file", "tree_index", "left", "right", "mid"])
        for rec in records:
            writer.writerow([stem, rec["tree_index"], rec["left"], rec["right"], rec["mid"]])

    print(f"Wrote {output_tree_path}")
    print(f"Wrote {output_csv_path}")
    print(f"Exported {len(records)} informative trees out of {ts_for_export.num_trees} total trees ({pop})")
    print(f"First exported interval: [{records[0]['left']}, {records[0]['right']})")
    print(f"Last exported interval : [{records[-1]['left']}, {records[-1]['right']})")

Wrote trees\chr2_52samples_CEU.tree
Wrote results\chr2_52samples_CEU_breaks.csv
Exported 50109 informative trees out of 50111 total trees (CEU)
First exported interval: [90000000.0, 90006187.0)
Last exported interval : [159999612.0, 160000000.0)
Wrote trees\chr2_52samples_GBR.tree
Wrote results\chr2_52samples_GBR_breaks.csv
Exported 49554 informative trees out of 49556 total trees (GBR)
First exported interval: [90000000.0, 90006187.0)
Last exported interval : [159999612.0, 160000000.0)
Wrote trees\chr2_52samples_TSI.tree
Wrote results\chr2_52samples_TSI_breaks.csv
Exported 53075 informative trees out of 53077 total trees (TSI)
First exported interval: [90000000.0, 90003264.0)
Last exported interval : [159999612.0, 160000000.0)
Wrote trees\chr2_52samples_JPT.tree
Wrote results\chr2_52samples_JPT_breaks.csv
Exported 50919 informative trees out of 50921 total trees (JPT)
First exported interval: [90000000.0, 90000900.0)
Last exported interval : [159999612.0, 160000000.0)
Wrote trees\chr2

# Sample 2 individuals per population (all populations) and export combined tree sequence

This chunk samples two individuals from every population present in the chromosome tree sequence, simplifies to those sampled nodes, and writes:

- `trees/chr{CHR}_2inds_allpops.trees`
- `trees/chr{CHR}_2inds_allpops.tree`
- `results/chr{CHR}_2inds_allpops_breaks.csv`
- `results/chr{CHR}_2inds_allpops_sample_manifest.csv`


In [7]:
# ----------------------------------------
# All-population sampling: 2 individuals per population (LCT interval only)
# ----------------------------------------
import csv

CHR_ALLPOPS_2 = int(CHR) if 'CHR' in globals() else 2
rng_seed_allpops_2inds = 20260228
lct_interval_allpops = [(90_000_000, 160_000_000)]

if 'ts' not in globals():
    ts_path_allpops = path_1000g(CHR_ALLPOPS_2)
    try:
        ts_allpops = tskit.load(str(ts_path_allpops))
    except Exception:
        import tszip
        ts_allpops = tszip.decompress(str(ts_path_allpops))
else:
    ts_allpops = ts

pop_meta_allpops = pd.DataFrame([
    json.loads(ts_allpops.population(i).metadata) for i in range(ts_allpops.num_populations)
])
pop_meta_allpops['pop_id'] = np.arange(ts_allpops.num_populations)
pop_meta_allpops = pop_meta_allpops.sort_values('name').reset_index(drop=True)

rng = np.random.default_rng(rng_seed_allpops_2inds)
selected_nodes = []
manifest_rows = []

for row in pop_meta_allpops.itertuples(index=False):
    pop_id = int(row.pop_id)
    pop_name = str(row.name)

    pop_nodes = np.array(ts_allpops.samples(pop_id), dtype=np.int32)
    if pop_nodes.size == 0:
        manifest_rows.append({
            'pop_id': pop_id,
            'population': pop_name,
            'available_individuals': 0,
            'sampled_individuals': 0,
            'sampled_nodes': 0,
            'mode': 'none',
        })
        continue

    node_individuals = ts_allpops.tables.nodes.individual[pop_nodes]
    indiv_ids = np.unique(node_individuals[node_individuals != tskit.NULL])

    if indiv_ids.size >= 2:
        chosen_indivs = rng.choice(indiv_ids, size=2, replace=False)
        chosen_indivs = np.array(chosen_indivs, dtype=np.int32)

        keep_mask = np.isin(node_individuals, chosen_indivs)
        chosen_nodes = pop_nodes[keep_mask]

        selected_nodes.extend(chosen_nodes.tolist())
        manifest_rows.append({
            'pop_id': pop_id,
            'population': pop_name,
            'available_individuals': int(indiv_ids.size),
            'sampled_individuals': int(chosen_indivs.size),
            'sampled_nodes': int(chosen_nodes.size),
            'mode': 'individuals',
        })
    else:
        # Fallback: if individuals are not encoded, sample up to 2 sample nodes.
        n_take = min(2, pop_nodes.size)
        chosen_nodes = rng.choice(pop_nodes, size=n_take, replace=False)
        chosen_nodes = np.array(chosen_nodes, dtype=np.int32)

        selected_nodes.extend(chosen_nodes.tolist())
        manifest_rows.append({
            'pop_id': pop_id,
            'population': pop_name,
            'available_individuals': int(indiv_ids.size),
            'sampled_individuals': int(0),
            'sampled_nodes': int(chosen_nodes.size),
            'mode': 'sample_nodes_fallback',
        })

selected_nodes = np.array(sorted(set(selected_nodes)), dtype=np.int32)
if selected_nodes.size == 0:
    raise RuntimeError('No samples selected for all-population 2-individual export.')

manifest_df = pd.DataFrame(manifest_rows).sort_values('population').reset_index(drop=True)

stem_allpops = f"chr{CHR_ALLPOPS_2}_2inds_allpops"
out_trees = trees_dir / f"{stem_allpops}.trees"
out_tree = trees_dir / f"{stem_allpops}.tree"
out_breaks = results_dir / f"{stem_allpops}_breaks.csv"
out_manifest = results_dir / f"{stem_allpops}_sample_manifest.csv"

# Mimic earlier workflow: keep interval first, then simplify sampled nodes.
ts_lct = ts_allpops.keep_intervals(lct_interval_allpops, simplify=False)
ts_sub = ts_lct.simplify(samples=selected_nodes)
ts_sub.dump(out_trees)

# Export Newick + breakpoints
if 'jitter_within_generations' not in globals():
    raise NameError('Expected jitter_within_generations to be defined in this notebook.')
if 'to_single_newick' not in globals():
    raise NameError('Expected to_single_newick to be defined in this notebook.')

ts_export = jitter_within_generations(ts_sub, eps=1e-9)
records = []
for tree_idx, tree in enumerate(ts_export.trees()):
    if tree.num_edges == 0:
        continue
    left, right = tree.interval.left, tree.interval.right
    records.append({
        'tree_index': int(tree_idx),
        'left': float(left),
        'right': float(right),
        'mid': float(0.5 * (left + right)),
        'newick': to_single_newick(tree),
    })

if len(records) == 0:
    raise RuntimeError('No informative trees found in all-population 2-individual subset.')

with open(out_tree, 'w') as f:
    for rec in records:
        f.write(rec['newick'] + '\n')

with open(out_breaks, 'w', newline='') as cf:
    writer = csv.writer(cf)
    writer.writerow(['file', 'tree_index', 'left', 'right', 'mid'])
    for rec in records:
        writer.writerow([stem_allpops, rec['tree_index'], rec['left'], rec['right'], rec['mid']])

manifest_df.to_csv(out_manifest, index=False)

print('Saved:', out_trees)
print('Saved:', out_tree)
print('Saved:', out_breaks)
print('Saved:', out_manifest)
print('Interval used:', lct_interval_allpops)
print('Total selected sample nodes:', selected_nodes.size)
print('Populations covered:', manifest_df.shape[0])
print(manifest_df[['population', 'sampled_individuals', 'sampled_nodes', 'mode']].head())


Saved: trees\chr2_2inds_allpops.trees
Saved: trees\chr2_2inds_allpops.tree
Saved: results\chr2_2inds_allpops_breaks.csv
Saved: results\chr2_2inds_allpops_sample_manifest.csv
Interval used: [(90000000, 160000000)]
Total selected sample nodes: 104
Populations covered: 26
  population  sampled_individuals  sampled_nodes         mode
0        ACB                    2              4  individuals
1        ASW                    2              4  individuals
2        BEB                    2              4  individuals
3        CDX                    2              4  individuals
4        CEU                    2              4  individuals
